# Ames Housing: Regression for House Prices

**Team:** Akhil, Veer, and Julie

## 1. Setup and getting the data

The data comes from the Kaggle competition as `ames.zip` (`train.csv`, `test.csv`).

In [ ]:
import os, glob, zipfile
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import RidgeCV, LassoCV
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# If train.csv is not here yet, upload ames.zip when prompted (runs only once per session)
if not os.path.exists('train.csv'):
    zips = glob.glob('ames*.zip')
    if not zips:
        from google.colab import files
        files.upload()
        zips = glob.glob('ames*.zip')
    with zipfile.ZipFile(zips[0]) as z:
        z.extractall('.')
print('Files:', [f for f in os.listdir('.') if f.endswith('.csv')])

## 2. Load data and remove outliers

We strip spaces from column names so they are easy to reference. We drop two houses with a very large living area (`GrLivArea > 4000`) but a low sale price: they are unusual cases that can pull a linear model off course.

We then take `log(1 + SalePrice)` as the target. Prices are right-skewed, and a log target makes errors more proportional and relationships closer to linear. Predictions are converted back to dollars with `expm1`.

In [ ]:
# ---------------------------------------------------------------- load data
def load(path):
    df = pd.read_csv(path)
    if df.shape[1] == 1:  # file was tab-separated
        df = pd.read_csv(path, sep='\t')
    df.columns = [c.replace(' ', '') for c in df.columns]  # 'Gr Liv Area' -> 'GrLivArea'
    return df

train = load('train.csv')
test = load('test.csv')

# Drop two well-known extreme outliers (huge houses sold cheap)
train = train[~((train['GrLivArea'] > 4000) & (train['SalePrice'] < 300000))].reset_index(drop=True)

y = train['SalePrice']
y_log = np.log1p(y)  # model log price, then convert back

## 3. Encoding, feature engineering and model setup

**Encoding**
- *Ordinal* columns (quality and condition ratings, basement exposure, garage finish, etc.) are mapped to ordered numbers, with `NA` (feature absent, e.g. no basement) as 0. Order is Po < Fa < TA < Gd < Ex.
- *Nominal* columns (e.g. `Neighborhood`) are one-hot encoded; `MSSubClass` is treated as a category because its numbers are codes.
- Missing numbers get the median, missing categories the most frequent value.

**Engineered features:** `TotalSF`, `Age`, `RemodAge`, `TotalBath`, `TotalPorchSF`, `HasPool`, `HasGarage`, and `OverallQual x GrLivArea`.

**Clipping test features (important lesson).** Our first Kaggle submission scored an RMSE of about **$175,000**, far worse than our cross-validation estimate of about $20,000. The cause was extrapolation: a linear model can produce an absurd price when a test row has an unusual value (such as a typo'd year). We now clip every test feature to the range seen in training, and clip final predictions to the training price range. That brought our score down to about **$54,000**.

**Models:** Ridge and Lasso (with scaled inputs) and a Gradient Boosting Regressor (trees need no scaling).

In [ ]:
# ---------------------------------------------------------- ordinal encoding
# NOTE: pandas reads the string "NA" as missing, so NaN here means "None/NA"
qual = {'NA': 0, 'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5}  # correct order: Po < Fa < TA < Gd < Ex
ordinal_maps = {
    'ExterQual': qual, 'ExterCond': qual, 'BsmtQual': qual, 'BsmtCond': qual,
    'HeatingQC': qual, 'KitchenQual': qual, 'FireplaceQu': qual,
    'GarageQual': qual, 'GarageCond': qual, 'PoolQC': qual,
    'BsmtExposure': {'NA': 0, 'No': 1, 'Mn': 2, 'Av': 3, 'Gd': 4},
    'BsmtFinType1': {'NA': 0, 'Unf': 1, 'LwQ': 2, 'Rec': 3, 'BLQ': 4, 'ALQ': 5, 'GLQ': 6},
    'BsmtFinType2': {'NA': 0, 'Unf': 1, 'LwQ': 2, 'Rec': 3, 'BLQ': 4, 'ALQ': 5, 'GLQ': 6},
    'GarageFinish': {'NA': 0, 'Unf': 1, 'RFn': 2, 'Fin': 3},
    'PavedDrive': {'N': 0, 'P': 1, 'Y': 2},
    'LotShape': {'IR3': 0, 'IR2': 1, 'IR1': 2, 'Reg': 3},
    'LandSlope': {'Sev': 0, 'Mod': 1, 'Gtl': 2},
    'CentralAir': {'N': 0, 'Y': 1},
    'Functional': {'Sal': 0, 'Sev': 1, 'Maj2': 2, 'Maj1': 3, 'Mod': 4, 'Min2': 5, 'Min1': 6, 'Typ': 7},
    'Fence': {'NA': 0, 'MnWw': 1, 'GdWo': 2, 'MnPrv': 3, 'GdPrv': 4},
}

def has(df, *cols):
    return all(c in df.columns for c in cols)

def prep(df):
    df = df.copy()
    for col, mapping in ordinal_maps.items():
        if col in df.columns:
            df[col] = df[col].fillna('NA').map(mapping)  # unseen -> NaN, imputed later

    # ------------------------------------------------ feature engineering
    if has(df, 'TotalBsmtSF', '1stFlrSF', '2ndFlrSF'):
        df['TotalSF'] = df['TotalBsmtSF'].fillna(0) + df['1stFlrSF'] + df['2ndFlrSF']
    if has(df, 'YrSold', 'YearBuilt'):
        df['Age'] = df['YrSold'] - df['YearBuilt']
    if has(df, 'YrSold', 'YearRemod/Add'):
        df['RemodAge'] = df['YrSold'] - df['YearRemod/Add']
    if has(df, 'FullBath', 'HalfBath', 'BsmtFullBath', 'BsmtHalfBath'):
        df['TotalBath'] = (df['FullBath'] + 0.5 * df['HalfBath']
                           + df['BsmtFullBath'].fillna(0) + 0.5 * df['BsmtHalfBath'].fillna(0))
    porch = [c for c in ['OpenPorchSF', 'EnclosedPorch', '3-SsnPorch', 'ScreenPorch', 'WoodDeckSF'] if c in df.columns]
    if porch:
        df['TotalPorchSF'] = df[porch].fillna(0).sum(axis=1)
    if has(df, 'OverallQual', 'GrLivArea'):
        df['QualxLiving'] = df['OverallQual'] * df['GrLivArea']
    if 'PoolArea' in df.columns:
        df['HasPool'] = (df['PoolArea'] > 0).astype(int)
    if 'GarageArea' in df.columns:
        df['HasGarage'] = (df['GarageArea'].fillna(0) > 0).astype(int)
    if 'MSSubClass' in df.columns:
        df['MSSubClass'] = df['MSSubClass'].astype(str)  # it's a category, not a number

    return df.drop(columns=[c for c in ['Id', 'PID', 'Order', 'SalePrice'] if c in df.columns])

X = prep(train)
X_test = prep(test)

num_cols = X.select_dtypes(include='number').columns.tolist()
cat_cols = X.select_dtypes(exclude='number').columns.tolist()

# Guard against extrapolation: clip test numbers to the range seen in training
# (e.g. a typo like GarageYrBlt = 2207 would otherwise blow up the linear models)
lo, hi = X[num_cols].min(), X[num_cols].max()
out = ((X_test[num_cols] < lo) | (X_test[num_cols] > hi)).sum()
print('Test values outside training range:\n', out[out > 0])
X_test[num_cols] = X_test[num_cols].clip(lower=lo, upper=hi, axis=1)

# Linear models: scale numbers, one-hot the rest
linear_pre = ColumnTransformer([
    ('num', Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler())]), num_cols),
    ('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')),
                      ('ohe', OneHotEncoder(handle_unknown='ignore'))]), cat_cols),
])
# Tree model: no scaling needed
tree_pre = ColumnTransformer([
    ('num', SimpleImputer(strategy='median'), num_cols),
    ('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')),
                      ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), cat_cols),
])

models = {
    'ridge': Pipeline([('pre', linear_pre), ('m', RidgeCV(alphas=np.logspace(-1, 3, 30)))]),
    'lasso': Pipeline([('pre', linear_pre), ('m', LassoCV(alphas=np.logspace(-5, -1, 30), max_iter=50000))]),
    'gbr': Pipeline([('pre', tree_pre), ('m', GradientBoostingRegressor(
        n_estimators=1500, learning_rate=0.02, max_depth=3, subsample=0.7,
        max_features=0.3, min_samples_leaf=5, random_state=42))]),
}

## 4. Model comparison with cross-validation

We score each model with 5-fold CV, converting predictions back to dollars so RMSE is comparable to Kaggle's metric. We also test a simple average of the three models.

Our results:

| Model | CV RMSE |
|---|---|
| Ridge | $20,607 |
| Lasso | $20,511 |
| Gradient Boosting | $21,805 |
| **Blend (average)** | **$19,724** |

The blend wins because the models make different kinds of errors and averaging cancels some of them out. Ridge and Lasso handle many correlated features well, while boosting captures non-linear effects.

In [ ]:
# ------------------------------------------------- compare models with 5-fold CV
def rmse(a, b):
    return np.sqrt(np.mean((a - b) ** 2))

kf = KFold(5, shuffle=True, random_state=42)
oof = {}
for name, pipe in models.items():
    oof[name] = np.expm1(cross_val_predict(pipe, X, y_log, cv=kf))
    print(f'{name:6s} CV RMSE: ${rmse(y, oof[name]):,.0f}')

oof['blend'] = np.mean([oof['ridge'], oof['lasso'], oof['gbr']], axis=0)
print(f'blend  CV RMSE: ${rmse(y, oof["blend"]):,.0f}')

best = min(oof, key=lambda k: rmse(y, oof[k]))
print('Best:', best)

## 5. Final fit and submission

We refit every model on all training data, predict on the test set, clip predictions to a sane range, and write `preds.csv` (`Id,SalePrice`, 260 rows; Kaggle expects Ids 0 to 259 in test-row order).

**Result:** our blend scored about **$54,120** RMSE on the public leaderboard (22nd of 28 at that point). _TODO: update with your final score and rank._

The gap between ~$20K in CV and ~$54K on Kaggle suggests some test houses are still predicted poorly (the public score also covers only part of the test set, so it is noisy).

In [ ]:
# ------------------------------------------------------------- fit & predict
for pipe in models.values():
    pipe.fit(X, y_log)
preds = {n: np.expm1(p.predict(X_test)) for n, p in models.items()}
preds['blend'] = np.mean([preds['ridge'], preds['lasso'], preds['gbr']], axis=0)
for k in preds:  # keep predictions inside the price range seen in training
    preds[k] = np.clip(preds[k], y.min(), y.max())
print(pd.Series(preds[best]).describe())

ids = range(len(test))   # Kaggle wants Ids 0..259 in test-row order

sub = pd.DataFrame({'Id': ids, 'SalePrice': preds[best]})
sub.to_csv('preds.csv', index=False)
print(sub.head())

# Sanity checks before uploading
print('Rows:', len(sub), '(expected 260)')
if len(sub) != 260:
    print('WARNING: row count is not 260 -- check test.csv was not filtered or altered.')
if sub['SalePrice'].isna().any() or (sub['SalePrice'] <= 0).any():
    print('WARNING: found missing or non-positive predictions.')

## 6. Download the predictions (Colab only)

In [ ]:
# ------------------------------------------------------------- download
try:
    from google.colab import files
    files.download('preds.csv')
except ImportError:
    pass